# 03 · Decision Network Model
**Project 18 — Decision Network for Intelligent Recommendation Systems**

This is the core notebook. We:
1. Construct the Bayesian Network structure (DAG)
2. Estimate Conditional Probability Tables (CPTs) via Bayesian Estimation
3. Perform Variable Elimination inference
4. Apply the Maximum Expected Utility (MEU) decision rule
5. Visualise the network, CPTs, and inference results

> **Decision Network = Bayesian Network + Utility Node + Decision Node**


In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
import warnings
warnings.filterwarnings('ignore')
from src.data_loader import load_all, train_test_split_temporal
from src.feature_engineering import build_user_profiles, build_item_profiles, build_dn_training_data
from src.decision_network import DecisionNetwork, EDGES, UTILITY_TABLE, EU_THRESHOLD


plt.rcParams.update({
    'figure.facecolor': '#0f1525', 'axes.facecolor': '#0f1525',
    'axes.edgecolor': '#2d3748',   'axes.labelcolor': '#a0aec0',
    'xtick.color': '#718096',      'ytick.color': '#718096',
    'text.color': '#e2e8f0',       'grid.color': '#1a2035',
})
PALETTE = ['#8b5cf6', '#06b6d4', '#10b981', '#f59e0b', '#f43f5e']
print('Libraries loaded ✓')


## 3.1  Load Data & Build Features

In [ ]:
ratings, users, items, merged = load_all()
train_r, test_r = train_test_split_temporal(ratings)
train_merged = merged[merged.index.isin(train_r.index)]

user_profiles = build_user_profiles(train_merged)
item_profiles  = build_item_profiles(train_merged)
dn_data        = build_dn_training_data(train_merged, user_profiles, item_profiles)

print(f'Training examples: {len(dn_data):,}')
print(f'Positive rate    : {dn_data["rating_positive"].astype(int).mean():.3f}')
dn_data.dtypes

## 3.2  Visualise the DAG Structure

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
ax.set_facecolor('#0f1525')
fig.patch.set_facecolor('#0f1525')

G = nx.DiGraph(EDGES)
pos = {
    'user_activity':      (0.0, 0.7),
    'user_rating_bucket': (0.0, 0.3),
    'user_gender':        (0.0, 0.0),
    'item_popularity':    (0.5, 0.9),
    'item_quality':       (0.5, 0.5),
    'item_genre':         (0.5, 0.1),
    'rating_positive':    (1.0, 0.5),
}

node_colors = {
    'user_activity':      '#8b5cf6', 'user_rating_bucket': '#8b5cf6',
    'user_gender':        '#8b5cf6', 'item_popularity':    '#06b6d4',
    'item_quality':       '#06b6d4', 'item_genre':         '#06b6d4',
    'rating_positive':    '#f59e0b',
}
colors = [node_colors[n] for n in G.nodes()]

nx.draw_networkx_nodes(G, pos, node_color=colors, node_size=2500, ax=ax, alpha=0.85)
nx.draw_networkx_edges(G, pos, edge_color='#a78bfa', ax=ax, arrows=True,
                       arrowsize=20, width=2, alpha=0.7,
                       connectionstyle='arc3,rad=0.1')
nx.draw_networkx_labels(G, pos, font_size=8, font_color='white',
                        font_weight='bold', ax=ax)

# Legend
legend_elements = [
    mpatches.Patch(facecolor='#8b5cf6', label='User Features (Chance Nodes)'),
    mpatches.Patch(facecolor='#06b6d4', label='Item Features (Chance Nodes)'),
    mpatches.Patch(facecolor='#f59e0b', label='rating_positive (Target)'),
]
ax.legend(handles=legend_elements, loc='lower right', facecolor='#1a2035',
          edgecolor='#2d3748', labelcolor='white', fontsize=9)
ax.set_title('Decision Network DAG — Conditional Dependencies', fontsize=13, color='#e2e8f0', pad=15)
ax.axis('off')

plt.tight_layout()
plt.savefig('../data/fig_dag_structure.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.3  Fit the Decision Network

In [ ]:
dn = DecisionNetwork(utility_threshold=EU_THRESHOLD)
dn.fit(dn_data, method='bayes')
print('\nModel CPDs:')
dn.print_cpds()

## 3.4  Variable Elimination Inference

In [ ]:
test_cases = [
    {
        'label'             : 'Generous active user + high-quality Action film',
        'user_activity'     : 'high',
        'user_rating_bucket': 'generous',
        'user_gender'       : 'M',
        'item_genre'        : 'Action',
        'item_popularity'   : 'popular',
        'item_quality'      : 'high',
    },
    {
        'label'             : 'Harsh critic + low-quality Horror film',
        'user_activity'     : 'low',
        'user_rating_bucket': 'harsh',
        'user_gender'       : 'F',
        'item_genre'        : 'Horror',
        'item_popularity'   : 'niche',
        'item_quality'      : 'low',
    },
    {
        'label'             : 'Neutral user + medium Romance',
        'user_activity'     : 'medium',
        'user_rating_bucket': 'neutral',
        'user_gender'       : 'F',
        'item_genre'        : 'Romance',
        'item_popularity'   : 'moderate',
        'item_quality'      : 'medium',
    },
]

print(f'{"Case":<50} {"P(like)":>8} {"EU":>8} {"Decision":>12}')
print('─' * 80)
for case in test_cases:
    ev = {k: v for k, v in case.items() if k != 'label'}
    decision, eu, prob = dn.decide(ev)
    print(f'{case["label"]:<50} {prob:>8.3f} {eu:>8.3f} {decision:>12}')

## 3.5  Sensitivity Analysis — EU vs. Item Quality

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# EU by quality × activity
ax = axes[0]
for activity, color in zip(['low','medium','high'], [PALETTE[4], PALETTE[0], PALETTE[2]]):
    eu_by_quality = []
    for quality in ['low','medium','high']:
        ev = {
            'user_activity':'medium', 'user_rating_bucket':'neutral',
            'user_gender':'M', 'item_genre':'Drama',
            'item_popularity':'moderate', 'item_quality': quality,
        }
        ev['user_activity'] = activity
        _, eu, _ = dn.decide(ev)
        eu_by_quality.append(eu)
    ax.plot(['low','medium','high'], eu_by_quality, 'o-', color=color,
            linewidth=2, label=f'activity={activity}', markersize=8)

ax.axhline(EU_THRESHOLD, color='white', linewidth=1.5, linestyle='--', alpha=0.6, label=f'Threshold θ={EU_THRESHOLD}')
ax.fill_between(['low','medium','high'], EU_THRESHOLD, 1.0,
                alpha=0.05, color='#10b981', label='Recommend zone')
ax.set_title('Expected Utility vs. Item Quality\n(Drama, neutral user)', color='#e2e8f0')
ax.set_xlabel('item_quality'); ax.set_ylabel('EU(recommend)')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# EU by genre (for generous user, high quality)
ax = axes[1]
genres = ['Action','Comedy','Drama','Horror','Romance','Sci-Fi','Thriller','Animation']
eu_by_genre = []
for g in genres:
    ev = {
        'user_activity':'high', 'user_rating_bucket':'generous',
        'user_gender':'M', 'item_genre': g,
        'item_popularity':'popular', 'item_quality':'high',
    }
    _, eu, _ = dn.decide(ev)
    eu_by_genre.append(eu)

colors_g = [PALETTE[2] if eu >= EU_THRESHOLD else PALETTE[4] for eu in eu_by_genre]
bars = ax.barh(genres, eu_by_genre, color=colors_g, alpha=0.85, edgecolor='none')
ax.axvline(EU_THRESHOLD, color='white', linewidth=1.5, linestyle='--', alpha=0.6, label=f'θ={EU_THRESHOLD}')
ax.set_title('EU by Genre\n(generous active user, high quality)', color='#e2e8f0')
ax.set_xlabel('EU(recommend)'); ax.legend(fontsize=9); ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../data/fig_sensitivity_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# Save model
dn.save('../data/decision_network_model.pkl')
print('\n✓ Notebook 03 complete. Model saved. Proceed to 04_evaluation_metrics.ipynb')